In [ ]:
from qualibrate import QualibrationNode, NodeParameters
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
node  = QualibrationNode(name="QPT_vs_ellipsoid")
QPT_ellipsoid_data_path = Path("data/gst_qpt_ellipsoid").resolve()

In [ ]:
from quam_libs.QPT_vs_ellipsoid_marcos import xyz_data_to_gate_fidelity, random_sampling
QPT_sampling_data = []
gate_fidelity = []
#QPT_data_id = np.arange(1824,1915,3)
# QPT_data_id = np.arange(1926,2017,3) # old data
QPT_data_id = np.arange(1005,1036,1) # new data

trials = 10
for i in QPT_data_id:
    QPT = node.load_from_id(i, base_path=QPT_ellipsoid_data_path)
    ds = QPT.results['ds']
    machine =  QPT.machine
    qubit = machine.qubits['q0']
    confusion_matrix = qubit.resonator.confusion_matrix
    n_runs = 10000  #maximum is 10000
    ds_samples = random_sampling(ds, n_samples=trials, sample_size= n_runs)
    for j in range(trials):
        results = xyz_data_to_gate_fidelity(
        ds_samples.sel(sample=j), 
        confusion_matrix,
        qubit,
        n_runs,
        apply_mitigation=True,
        operation_name='id'
        )
        QPT_sampling_data.append(results['q0']['quantum_information']['mitigated']['Quantum Memory Robustness'])
        gate_fidelity.append(results['q0']['quantum_information']['mitigated']['fidelity'])
QPT_sampling_data, gate_fidelity = np.array(QPT_sampling_data), np.array(gate_fidelity)
QPT_sampling_data = QPT_sampling_data.reshape(len(QPT_data_id),trials)
gate_fidelity = gate_fidelity.reshape(len(QPT_data_id),trials)

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial

# Extract mean data for fitting
x_data = gate_fidelity.mean(axis=1)
QPT_y_data = QPT_sampling_data.mean(axis=1)
QPT_y_err = QPT_sampling_data.std(axis=1)

# Fit with polynomial (1st order = linear fit)
p = Polynomial.fit(x_data, QPT_y_data, 1)
QPT_x_fit = np.linspace(x_data.min(), x_data.max()+0.1, 100)
QPT_y_fit = p(QPT_x_fit)

# Plot data points with green color and error bars
plt.errorbar(x_data, QPT_y_data, QPT_y_err,
                    linestyle='none', fmt='o', markersize=4, markerfacecolor='green', 
                    markeredgecolor='green', ecolor='green', elinewidth=2, capsize=4, capthick=1.2)

# Plot fitting line as dashed line
plt.plot(QPT_x_fit, QPT_y_fit, 'g--', linewidth=2, label='Linear Fit')
plt.xlim(0.85,1.0)
plt.ylim(0.5,0.9)
plt.xlabel('Gate Fidelity(%)')
plt.ylabel('Robustness')
plt.show()

# ellipsoid

In [ ]:
from quam_libs.QPT_vs_ellipsoid_marcos import random_sampling_without_replacement, compute_robustness, filter_robustness_outliers

ellipsoid_data_id = np.arange(1036,1067,1) # new
number_of_ellipsoid =10
data_points_per_ellipsoid = 180
robustness_results_all = []  # Save all original robustness values
robustness_results_filtered = []  # Save filtered robustness values
filter_stats = []  # Save filtering statistics for each ID
ellipsoid_statistics = {}  # Dictionary to store mean and std for each ID

for i in range(len(ellipsoid_data_id)):
    QPT = node.load_from_id(ellipsoid_data_id[i], base_path=QPT_ellipsoid_data_path)
    ds = QPT.results['ds']
    machine =  QPT.machine
    qubit = machine.qubits['q0']
    confusion_matrix = qubit.resonator.confusion_matrix
    ds_samples = random_sampling_without_replacement(ds, n_samples=number_of_ellipsoid, sample_size=data_points_per_ellipsoid)
    
    # Compute robustness for all samples
    robustness_per_id = []
    for j in range(number_of_ellipsoid):
        robustness, results = compute_robustness(ds_samples[j], np.array(confusion_matrix).T, apply_mitigation=True)
        robustness_per_id.append(robustness)
        robustness_results_all.append(robustness)
    
    # Filter outliers using IQR method for this ID
    robustness_array = np.array(robustness_per_id)
    filtered_array, mask, stats = filter_robustness_outliers(
        robustness_array, 
        method='iqr', 
        threshold=1.5,
        verbose=False
    )
    
    robustness_results_filtered.extend(filtered_array)
    stats['ellipsoid_id'] = ellipsoid_data_id[i]
    filter_stats.append(stats)
    
    # Store statistics for this ID
    mean_robustness = np.mean(filtered_array)
    std_robustness = np.std(filtered_array)
    ellipsoid_statistics[int(ellipsoid_data_id[i])] = {
        'mean': mean_robustness,
        'std': std_robustness,
        'count': len(filtered_array),
        'outliers_removed': stats['removed_count']
    }
    
    print(f"ID {ellipsoid_data_id[i]}: Retained {stats['filtered_count']}/{stats['original_count']} data "
          f"(removed {stats['removed_count']}, {stats['removed_percentage']:.1f}%) | "
          f"Mean: {mean_robustness:.6f} ± {std_robustness:.6f}")

robustness_results_all = np.array(robustness_results_all)
robustness_results_filtered = np.array(robustness_results_filtered)
robustness_results = robustness_results_filtered  # Use filtered results

print(f"\nOverall Statistics:")
print(f"  Original data: {len(robustness_results_all)} points")
print(f"  Filtered data: {len(robustness_results_filtered)} points")
print(f"  Removed: {len(robustness_results_all) - len(robustness_results_filtered)} points ({100*(1-len(robustness_results_filtered)/len(robustness_results_all)):.1f}%)")

print(f"\nEllipsoid Statistics Dictionary:")
print(ellipsoid_statistics)

In [ ]:
ellipsoid_mean = []
ellipsoid_std = []
for i in ellipsoid_data_id:
    ellipsoid_mean.append(ellipsoid_statistics[i]['mean'])
    ellipsoid_std.append(ellipsoid_statistics[i]['std'])

In [ ]:
# Extract mean data for fitting
x_data = gate_fidelity.mean(axis=1)
ellipsoid_y_data = ellipsoid_mean
ellipsoid_y_err = ellipsoid_std

# Fit with polynomial (1st order = linear fit)
p = Polynomial.fit(x_data, ellipsoid_y_data, 1)
ellipsoid_x_fit = np.linspace(x_data.min(), x_data.max()+0.1, 100)
ellipsoid_y_fit = p(ellipsoid_x_fit)

# Plot data points with green color and error bars
plt.errorbar(x_data, ellipsoid_y_data, ellipsoid_y_err,
                    linestyle='none', fmt='o', markersize=4, markerfacecolor='blue', 
                    markeredgecolor='blue', ecolor='blue', elinewidth=2, capsize=4, capthick=1.2)

# Plot fitting line as dashed line
plt.plot(ellipsoid_x_fit, ellipsoid_y_fit, 'b--', linewidth=2, label='Linear Fit')
plt.xlim(0.85,1.0)
plt.ylim(0.7,1)
plt.xlabel('Gate Fidelity(%)')
plt.ylabel('Robustness')
plt.show()

# GST

In [1]:
from qualibrate import QualibrationNode, NodeParameters
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

node  = QualibrationNode(name="GST")
QPT_ellipsoid_data_path = Path("data/gst_qpt_ellipsoid").resolve()
from quam_libs.QPT_vs_ellipsoid_marcos import run_gst_monte_carlo
import pygsti
from pygsti.modelpacks import smq1Q_XYI as std

2026-03-25 16:29:21,319 - qm - INFO     - Starting session: 74aa4ea9-e7e8-4b65-85aa-5e9d4fe553f0


2026-03-25 16:29:22,798 - qualibrate - INFO - Creating node GST
/Users/thwuedwin/lab/quantum_memory_data/.venv/lib/python3.12/site-packages/pennylane/__init__.py:212: PennyLaneDeprecationWarning: PennyLane v0.44 has dropped maintainence support for NumPy < 2.0.0. You have version 1.26.4 installed. Future versions of PennyLane will not work with NumPy<2.0. Please consider upgrading NumPy using `python -m pip install numpy --upgrade`. 
  warnings.warn(


In [ ]:
GST_data_id = np.arange(1098, 1129,1) 
trials = 5
max_circuit_length = [1, 2, 4, 8]
std_model = std.target_model()
exp_design = pygsti.protocols.StandardGSTDesign(
    std_model, 
    std.prep_fiducials(), 
    std.meas_fiducials(), 
    std.germs(),
    max_circuit_length
)
gst_mean_rob = []
gst_std_rob = []
for i in GST_data_id:
    GST = node.load_from_id(i, base_path=QPT_ellipsoid_data_path)
    ds = GST.results['ds'].sel(qubit='q0')
    data = run_gst_monte_carlo(ds, exp_design, num_resamples=100, n_shots=1000)
    mean_rob, std_rob, mc_robustness_results, simulated_counts1  = data
    gst_mean_rob.append(mean_rob)
    gst_std_rob.append(std_rob)
    print(f"Node {i}/31 completed.")

Starting Monte Carlo analysis with 100 resamples...


/Users/thwuedwin/lab/quantum_memory_data/.venv/lib/python3.12/site-packages/pygsti/modelmembers/povms/__init__.py:615: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  errgen_vec = _np.linalg.lstsq(phys_directions, soln.x)[0]


    Iteration 10/100 completed.
    Iteration 20/100 completed.
    Iteration 30/100 completed.


/Users/thwuedwin/lab/quantum_memory_data/.venv/lib/python3.12/site-packages/pygsti/objectivefns/objectivefns.py:4502: RuntimeWarning: divide by zero encountered in divide
  p5over_lsvec = 0.5/lsvec


    Iteration 40/100 completed.
    Iteration 50/100 completed.
    Iteration 60/100 completed.
    Iteration 70/100 completed.
    Iteration 80/100 completed.
    Iteration 90/100 completed.
    Iteration 100/100 completed.
Node 1098/31 completed.
Starting Monte Carlo analysis with 100 resamples...


...

    Iteration 30/100 completed.
    Iteration 40/100 completed.
    Iteration 50/100 completed.
    Iteration 60/100 completed.
    Iteration 70/100 completed.
    Iteration 80/100 completed.
    Iteration 90/100 completed.
    Iteration 100/100 completed.
Node 1115/31 completed.
Starting Monte Carlo analysis with 100 resamples...


/Users/thwuedwin/lab/quantum_memory_data/.venv/lib/python3.12/site-packages/pygsti/modelmembers/povms/__init__.py:615: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  errgen_vec = _np.linalg.lstsq(phys_directions, soln.x)[0]


    Iteration 10/100 completed.
    Iteration 20/100 completed.
    Iteration 30/100 completed.


/Users/thwuedwin/lab/quantum_memory_data/.venv/lib/python3.12/site-packages/pygsti/objectivefns/objectivefns.py:4502: RuntimeWarning: divide by zero encountered in divide
  p5over_lsvec = 0.5/lsvec


    Iteration 40/100 completed.
    Iteration 50/100 completed.
    Iteration 60/100 completed.
    Iteration 70/100 completed.
    Iteration 80/100 completed.
    Iteration 90/100 completed.


In [ ]:
import pandas as pd
csv_path = 'QPT_vs_Ellipsoid_data_new.csv'
df = pd.read_csv(csv_path)
df['GST robustness'] = gst_mean_rob
df['GST error bar'] = gst_std_rob
df.to_csv(csv_path, index=False)

# Plot together

In [ ]:
from qualibrate import QualibrationNode, NodeParameters
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
csv_path = 'QPT_vs_Ellipsoid_data_new.csv'
df = pd.read_csv(csv_path)
df = df.sort_values(by='gate fidelity', ascendind=False)

In [ ]:
fig, ax = plt.subplot(1, 1, figsize=(8, 6))
xdata = df['gate fidelity']
ax.errorbar(
    xdata, df['ellipsoid robustness'], df['ellipsoid error bar']
    linestyle='none', fmt='o', markersize=4, markerfacecolor='blue', 
    markeredgecolor='blue', ecolor='blue', elinewidth=2, capsize=4, capthick=1.2
)
ax.errorbar(
    xdata, df['QPT robustness'], df['QPT error bar'],
    linestyle='none', fmt='o', markersize=4, markerfacecolor='green', 
    markeredgecolor='green', ecolor='green', elinewidth=2, capsize=4, capthick=1.2
)
ax.errorbar(
    xdata, df['GST robustness'], df['GST error bar'],
    linestyle='none', fmt='o', markersize=4, markerfacecolor='blue', 
    markeredgecolor='red', ecolor='red', elinewidth=2, capsize=4, capthick=1.2
)

plt.xlim(0.85,1.0)
plt.ylim(0.7,1)
plt.xlabel('Gate Fidelity(%)')
plt.ylabel('Robustness')
plt.show()

In [ ]:
# Plot data points with green color and error bars
plt.errorbar(x_data, QPT_y_data, QPT_y_err,
                    linestyle='none', fmt='o', markersize=4, markerfacecolor='green', 
                    markeredgecolor='green', ecolor='green', elinewidth=2, capsize=4, capthick=1.2)

# Plot fitting line as dashed line
plt.plot(QPT_x_fit, QPT_y_fit, 'g--', linewidth=2, label='Linear Fit')

# Plot data points with green color and error bars
plt.errorbar(x_data, ellipsoid_y_data, ellipsoid_y_err,
                    linestyle='none', fmt='o', markersize=4, markerfacecolor='blue', 
                    markeredgecolor='blue', ecolor='blue', elinewidth=2, capsize=4, capthick=1.2)

# Plot fitting line as dashed line
plt.plot(ellipsoid_x_fit, ellipsoid_y_fit, 'b--', linewidth=2, label='Linear Fit')

plt.xlim(0.85,1.0)
plt.ylim(0.7,1)
plt.xlabel('Gate Fidelity(%)')
plt.ylabel('Robustness')
plt.show()

In [ ]:
# Filter both QPT and ellipsoid data based on ellipsoid error bars
ellipsoid_y_err_array = np.array(ellipsoid_y_err)
x_data_array = np.array(x_data)
ellipsoid_y_data_array = np.array(ellipsoid_y_data)
QPT_y_data_array = np.array(QPT_y_data)
QPT_y_err_array = np.array(QPT_y_err)

# Define threshold for error bar - keep points with std below the median + 0.45 std dev
err_threshold = np.median(ellipsoid_y_err_array) + 0.45*np.std(ellipsoid_y_err_array)
print(f"Error bar threshold (based on ellipsoid): {err_threshold:.6f}")
print(f"Median error: {np.median(ellipsoid_y_err_array):.6f}")
print(f"Std of errors: {np.std(ellipsoid_y_err_array):.6f}")

# Filter mask based on ellipsoid error bars
mask = ellipsoid_y_err_array <= err_threshold

# Apply same mask to both datasets
filtered_x = x_data_array[mask]
filtered_ellipsoid_y = ellipsoid_y_data_array[mask]
filtered_ellipsoid_err = ellipsoid_y_err_array[mask]
filtered_QPT_y = QPT_y_data_array[mask]
filtered_QPT_err = QPT_y_err_array[mask]

print(f"\nFiltering results:")
print(f"  Original data points: {len(x_data_array)}")
print(f"  Filtered data points: {len(filtered_x)}")
print(f"  Removed: {len(x_data_array) - len(filtered_x)} ({100*(1-len(filtered_x)/len(x_data_array)):.1f}%)")

# Fit both datasets with the filtered data
p_QPT_filtered = Polynomial.fit(filtered_x, filtered_QPT_y, 1)
p_ellipsoid_filtered = Polynomial.fit(filtered_x, filtered_ellipsoid_y, 1)
x_fit_filtered = np.linspace(x_data_array.min(), x_data_array.max()+0.1, 100)
QPT_y_fit_filtered = p_QPT_filtered(x_fit_filtered)
ellipsoid_y_fit_filtered = p_ellipsoid_filtered(x_fit_filtered)

print(f"\nFit comparison (Original vs Filtered):")
print(f"  QPT:")
print(f"    Original:  y = {p.coef[1]:.4f}*x + {p.coef[0]:.4f}")
print(f"    Filtered:  y = {p_QPT_filtered.coef[1]:.4f}*x + {p_QPT_filtered.coef[0]:.4f}")
print(f"  Ellipsoid:")
print(f"    Original:  y = {p.coef[1]:.4f}*x + {p.coef[0]:.4f}")
print(f"    Filtered:  y = {p_ellipsoid_filtered.coef[1]:.4f}*x + {p_ellipsoid_filtered.coef[0]:.4f}")

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original data
ax1.errorbar(x_data, QPT_y_data, QPT_y_err,
                    linestyle='none', fmt='o', markersize=4, markerfacecolor='green', 
                    markeredgecolor='green', ecolor='green', elinewidth=2, capsize=4, capthick=1.2,
                    label='QPT Data')
ax1.plot(QPT_x_fit, QPT_y_fit, 'g--', linewidth=2, label='QPT Fit')

ax1.errorbar(x_data, ellipsoid_y_data, ellipsoid_y_err,
                    linestyle='none', fmt='s', markersize=4, markerfacecolor='blue', 
                    markeredgecolor='blue', ecolor='blue', elinewidth=2, capsize=4, capthick=1.2,
                    label='Ellipsoid Data (All)')
ax1.plot(ellipsoid_x_fit, ellipsoid_y_fit, 'b--', linewidth=2, label='Ellipsoid Fit (All)')

ax1.set_xlim(0.85, 1.0)
ax1.set_ylim(0.7, 1)
ax1.set_xlabel('Gate Fidelity', fontsize=11)
ax1.set_ylabel('Robustness', fontsize=11)
ax1.set_title('Original Data (All Points)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=9)

# Filtered data (both QPT and ellipsoid)
ax2.errorbar(filtered_x, filtered_QPT_y, filtered_QPT_err,
                    linestyle='none', fmt='o', markersize=5, markerfacecolor='green', 
                    markeredgecolor='green', ecolor='green', elinewidth=2, capsize=4, capthick=1.2,
                    label='QPT Data (Filtered)')
ax2.plot(x_fit_filtered, QPT_y_fit_filtered, 'g--', linewidth=2, label='QPT Fit (Filtered)')

ax2.errorbar(filtered_x, filtered_ellipsoid_y, filtered_ellipsoid_err,
                    linestyle='none', fmt='s', markersize=5, markerfacecolor='red', 
                    markeredgecolor='red', ecolor='red', elinewidth=2, capsize=4, capthick=1.2,
                    label='Ellipsoid Data (Filtered)')
ax2.plot(x_fit_filtered, ellipsoid_y_fit_filtered, 'r--', linewidth=2, label='Ellipsoid Fit (Filtered)')

# Show removed points in gray
removed_mask = ~mask
ax2.errorbar(x_data_array[removed_mask], ellipsoid_y_data_array[removed_mask], ellipsoid_y_err_array[removed_mask],
                    linestyle='none', fmt='x', markersize=6, color='gray', ecolor='gray', elinewidth=1, capsize=3,
                    alpha=0.5, label='Removed Points (by ellipsoid error bar)')

ax2.set_xlim(0.85, 1.0)
ax2.set_ylim(0.7, 1)
ax2.set_xlabel('Gate Fidelity', fontsize=11)
ax2.set_ylabel('Robustness', fontsize=11)
ax2.set_title('Filtered Data (Removed by Ellipsoid Error Bar)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nFiltered data statistics:")
print(f"  QPT (filtered) - Mean: {filtered_QPT_y.mean():.6f}, Std: {filtered_QPT_err.mean():.6f}")
print(f"  Ellipsoid (filtered) - Mean: {filtered_ellipsoid_y.mean():.6f}, Std: {filtered_ellipsoid_err.mean():.6f}")

In [ ]:
filtered_x = 100*filtered_x  
x_fit_filtered = 100*x_fit_filtered

In [ ]:
fig, ax2 = plt.subplots(1, 1, figsize=(8, 6))

# Filtered data (both QPT and ellipsoid)
ax2.errorbar(filtered_x, filtered_QPT_y, filtered_QPT_err,
                    linestyle='none', fmt='o', markersize=5, markerfacecolor='green', 
                    markeredgecolor='green', ecolor='green', elinewidth=2, capsize=4, capthick=1.2,
                    label='QPT Data')
ax2.plot(x_fit_filtered, QPT_y_fit_filtered, 'g--', linewidth=2, label='QPT Fit')

ax2.errorbar(filtered_x, filtered_ellipsoid_y, filtered_ellipsoid_err,
                    linestyle='none', fmt='s', markersize=5, markerfacecolor='red', 
                    markeredgecolor='red', ecolor='red', elinewidth=2, capsize=4, capthick=1.2,
                    label='Ellipsoid Data')
ax2.plot(x_fit_filtered, ellipsoid_y_fit_filtered, 'r--', linewidth=2, label='Ellipsoid Fit')

ax2.set_xlim(85, 100)
# ax2.set_ylim(0.9, 1)
ax2.set_xlabel('Gate Fidelity (%)', fontsize=16)
ax2.set_ylabel('Robustness', fontsize=16)
ax2.tick_params(axis='x', labelsize=13) 
ax2.tick_params(axis='y', labelsize=13) 
# ax2.set_xticks([86,89,92,95,98])
# ax2.set_yticks([0.74,0.81, 0.89,0.97])
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=14)

plt.tight_layout()
plt.show()

# Print fitted results
print(f"\n{'='*60}")
print(f"Fitted Results (Filtered Data)")
print(f"{'='*60}")
print(f"\nQPT (x in percentage scale [85, 100]):")
a_QPT = p_QPT_filtered.coef[1] / 100
b_QPT = p_QPT_filtered.coef[0]
print(f"  y = {a_QPT:.6f} * x + {b_QPT:.6f}")
print(f"  OR: y = {p_QPT_filtered.coef[1]:.6f} * x + {b_QPT:.6f}  (x in [0, 1])")

print(f"\nEllipsoid (x in percentage scale [85, 100]):")
a_ellipsoid = p_ellipsoid_filtered.coef[1] / 100
b_ellipsoid = p_ellipsoid_filtered.coef[0]
print(f"  y = {a_ellipsoid:.6f} * x + {b_ellipsoid:.6f}")
print(f"  OR: y = {p_ellipsoid_filtered.coef[1]:.6f} * x + {b_ellipsoid:.6f}  (x in [0, 1])")
print(f"{'='*60}\n")